# Máquinas Térmicas — Módulo Bonus ⚛️
## Reacciones nucleares: fisión y fusión


### Objetivos
1. Conectar el **defecto de masa** ($E=mc^2$) y la **energía de enlace por nucleón** con la energía liberada en fisión y fusión.
2. Cuantificar la **fisión** de $^{235}$U (energía por evento, consumo de combustible, cinética del reactor).
3. Cuantificar la **fusión** D-T (barrera de Coulomb, factor de Gamow, reactividad $\langle\sigma v\rangle$, potencia del plasma).
4. **Comparar** ambas —y con la combustión química— por reacción, por nucleón y por kilogramo.


## 1. Contexto: por qué el núcleo libera millones de veces más energía
En la Lección 5 vimos que la energía de enlace por nucleón $B/A$ crece hasta el **hierro-56** y luego decrece. De ahí salen **dos formas** de liberar energía nuclear:

- **Fisión:** partir un núcleo **pesado** (p. ej. $^{235}$U) en dos fragmentos más cercanos al pico de $B/A$.
- **Fusión:** unir núcleos **ligeros** (p. ej. deuterio + tritio) para subir hacia el pico.

En ambos casos, la masa de los productos es **menor** que la de los reactivos; ese defecto de masa $\Delta m$ se convierte en energía por $E=\Delta m\,c^2$. El resultado: ~200 MeV por fisión y ~17.6 MeV por fusión, frente a los pocos **eV** de un enlace **químico** (Lección 5). Un factor de **un millón**.


## 2. Fundamento
**Equivalencia masa–energía** y **energía de enlace** de un núcleo con $Z$ protones y $N$ neutrones:

$$E=\Delta m\,c^{2},\qquad B(Z,N)=\big[Z\,m_p+N\,m_n-m_{\text{núcleo}}\big]c^{2}.$$

**Fisión** del uranio-235 (reacción representativa):

$$^{235}\text{U} + n \;\rightarrow\; \text{2 fragmentos} + \bar\nu\,n + \text{energía}\;(\approx 200\ \text{MeV}),\qquad \bar\nu\approx 2.43.$$

**Fusión** deuterio–tritio (la más accesible en la Tierra):

$$^{2}\text{H} + {}^{3}\text{H} \;\rightarrow\; {}^{4}\text{He}\,(3.5\ \text{MeV}) + n\,(14.1\ \text{MeV}),\qquad Q=17.6\ \text{MeV}.$$

Para fusionar, los núcleos deben vencer la **barrera de Coulomb** $V_C = \dfrac{Z_1Z_2e^2}{4\pi\varepsilon_0 r}$ por **efecto túnel** (factor de Gamow), lo que exige temperaturas de plasma de decenas de keV. La probabilidad de reacción se resume en la **reactividad** $\langle\sigma v\rangle(T)$.


In [1]:
# Instalación solo en Google Colab; en el entorno local del curso es un no-op
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "scipy", "ipywidgets"], check=True)


In [2]:
# === Setup: constantes físicas y datos de las reacciones ====================
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

J_per_MeV = 1.602176634e-13
NA = 6.02214076e23
u_kg = 1.66053907e-27
c = 2.99792458e8

# --- Fisión U-235 (valores de la fuente estudiantil) ---
E_FIS_MeV = 200.0
FIS_DESGLOSE = {"Fragmentos (Ec)": 168.0, "Neutrones (Ec)": 7.0, "γ prompt": 5.0,
                "β/γ retardados": 12.0, "Antineutrinos": 8.0}   # suma = 200
NU_BAR = 2.43
M_U235 = 235.0
# --- Fusión D-T ---
Q_FUS_MeV = 17.6
FUS_DESGLOSE = {"He (α, Ec)": 3.5, "Neutrón (Ec)": 14.1}         # suma = 17.6
M_D, M_T = 2.014, 3.016
# reactividad Maxwelliana D-T (ajuste de la fuente; Conn & Wesson) [m³/s], T en keV
def sigma_v_DT(T_keV):
    return 1.1e-24 * T_keV**2 * np.exp(-19.94 / T_keV)

# --- Combustión química de referencia (Lecciones 4-5) ---
LHV_CH4_MJkg = 50.0   # poder calorífico del metano ≈ 50 MJ/kg

def energia_por_kg(E_MeV, masa_u):
    """Energía específica [J/kg] de una reacción nuclear."""
    return (E_MeV * J_per_MeV) / (masa_u * u_kg)

print("Setup OK — fisión: %.0f MeV, fusión D-T: %.1f MeV" % (E_FIS_MeV, Q_FUS_MeV))


Setup OK — fisión: 200 MeV, fusión D-T: 17.6 MeV


## 3. Modelo computacional y widgets

### 3.1 Ancla: energía de enlace por nucleón (fisión ⟷ fusión)


In [3]:
# WIDGET 1 — B/A por nucleón, señalando fisión (pesado) y fusión (ligero)
def _BA(Z, A):
    aV, aS, aC, aA, aP = 15.75, 17.8, 0.711, 23.7, 11.18
    if A < 1: return 0.0
    delta = aP/A**0.5 if (Z % 2 == 0 and (A-Z) % 2 == 0) else \
            (-aP/A**0.5 if (Z % 2 == 1 and (A-Z) % 2 == 1) else 0.0)
    return (aV*A - aS*A**(2/3) - aC*Z*(Z-1)/A**(1/3) - aA*(A-2*Z)**2/A + delta)/A

def ancla_binding(A_fusion=5, A_fision=235):
    As = np.arange(2, 250)
    Zs = np.array([max(1, round(a/(1.98+0.015*a**(2/3)))) for a in As])
    ba = np.array([_BA(z, a) for z, a in zip(Zs, As)])
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(As, ba, color="#2554E8", lw=2)
    ax.axvline(56, ls="--", color="#B91C1C", lw=1); ax.text(58, 8.9, "Fe-56", color="#B91C1C")
    for A0, col, lab in [(A_fusion, "#27AE60", "fusión"), (A_fision, "#C0392B", "fisión")]:
        z0 = max(1, round(A0/(1.98+0.015*A0**(2/3))))
        ax.scatter([A0], [_BA(z0, A0)], s=90, color=col, edgecolors="k", zorder=5)
    ax.annotate("fusión: sube hacia el pico\n(gana ≈ energía × A)", (A_fusion, _BA(2, A_fusion)),
                xytext=(40, 3), fontsize=9, color="#27AE60",
                arrowprops=dict(arrowstyle="-|>", color="#27AE60"))
    ax.annotate("fisión: se parte\nhacia el pico", (A_fision, _BA(92, A_fision)),
                xytext=(150, 5.5), fontsize=9, color="#C0392B",
                arrowprops=dict(arrowstyle="-|>", color="#C0392B"))
    ax.fill_betweenx([0, 9.4], 2, 56, alpha=0.05, color="green")
    ax.fill_betweenx([0, 9.4], 56, 250, alpha=0.05, color="orange")
    ax.set_xlabel("Número másico A"); ax.set_ylabel("B/A [MeV/nucleón]")
    ax.set_ylim(0, 9.4); ax.set_title("Ambas reacciones buscan el máximo de energía de enlace")
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

interact(ancla_binding,
         A_fusion=IntSlider(5, min=2, max=20, step=1, description="A fusión"),
         A_fision=IntSlider(235, min=150, max=245, step=1, description="A fisión"));


interactive(children=(IntSlider(value=5, description='A fusión', max=20, min=2), IntSlider(value=235, descript…

### 3.2 Reactor de fisión: energía y consumo de combustible


In [4]:
# WIDGET 2 — reactor de fisión: fisiones/s, consumo de U-235 y potencia eléctrica
def reactor_fision(MWt=3000.0, eta=0.33, frac_recuperable=0.90):
    E_f = E_FIS_MeV * J_per_MeV                 # J/fisión (total)
    Pth = MWt * 1e6                             # W térmicos
    R_f = Pth / (frac_recuperable * E_f)        # fisiones/s (solo cuenta el calor útil)
    R_n = NU_BAR * R_f
    m_por_fision = M_U235 / 1000 / NA           # kg por núcleo
    burn_kg_dia = R_f * m_por_fision * 86400
    MWe = MWt * eta
    E_kg = energia_por_kg(E_FIS_MeV, M_U235)    # J/kg
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
    labels = list(FIS_DESGLOSE); vals = list(FIS_DESGLOSE.values())
    colores = ["#C0392B", "#E67E22", "#F1C40F", "#8E44AD", "#7F8C8D"]
    ax1.bar(labels, vals, color=colores)
    ax1.set_ylabel("MeV por fisión"); ax1.set_title("Desglose energético (U-235)")
    ax1.tick_params(axis="x", rotation=20); ax1.grid(axis="y", alpha=0.3)
    ax1.axhline(0)
    ax2.axis("off")
    ficha = (f"$\\bf{{Reactor\\ PWR}}$\n\n"
             f"Potencia térmica: {MWt:,.0f} MWt\n"
             f"Potencia eléctrica: {MWe:,.0f} MWe (η={eta:.0%})\n\n"
             f"Energía útil/fisión: {frac_recuperable*E_FIS_MeV:.0f} MeV\n"
             f"Fisiones por segundo: {R_f:.2e} s⁻¹\n"
             f"Neutrones/s (ν̄={NU_BAR}): {R_n:.2e} s⁻¹\n\n"
             f"Consumo $^{{235}}$U: {burn_kg_dia:.2f} kg/día\n"
             f"Energía específica: {E_kg/1e12:.0f} TJ/kg")
    ax2.text(0.03, 0.5, ficha, va="center", fontsize=11,
             bbox=dict(boxstyle="round", fc="#FEF9E7", ec="#999"))
    plt.tight_layout(); plt.show()

interact(reactor_fision,
         MWt=FloatSlider(3000, min=500, max=4500, step=100, description="MWt"),
         eta=FloatSlider(0.33, min=0.28, max=0.40, step=0.01, description="η"),
         frac_recuperable=FloatSlider(0.90, min=0.80, max=1.0, step=0.01, description="frac. útil"));


interactive(children=(FloatSlider(value=3000.0, description='MWt', max=4500.0, min=500.0, step=100.0), FloatSl…

### 3.3 Cinética puntual del reactor (control de la reacción en cadena)
>Modelo de **cinética puntual con 6 grupos de neutrones retardados** (constantes de Keepin) y realimentación Doppler:
>
> $$\frac{dn}{dt}=\frac{\rho-\beta}{\ell}\,n+\sum_{i=1}^{6}\lambda_i C_i,\qquad \frac{dC_i}{dt}=\frac{\beta_i}{\ell}\,n-\lambda_i C_i.$$
>
> Los **neutrones retardados** ($\beta=0.0065$) son los que hacen **controlable** el reactor: mientras $\rho<\beta$ la potencia sube despacio; si $\rho\ge\beta$ el reactor es *pronto-crítico* y se dispara.


In [5]:
# WIDGET 3 — transitorio de potencia del reactor según la reactividad ρ
BETA_I = np.array([0.000215, 0.001424, 0.001274, 0.002568, 0.000748, 0.000273])
LAMBDA_I = np.array([0.0124, 0.0305, 0.1110, 0.3010, 1.1400, 3.0100])
BETA = BETA_I.sum()          # 0.0065
ELL = 5e-5                   # tiempo prompt [s]
ALPHA_DOPPLER = -3e-5        # Δρ/K
WTH, MCP, UA, TCOOL, T0 = 3000e6, 5e10, 5e7, 580.0, 600.0

def _kin(t, y, rho_ext, feedback):
    n = y[0]; Ci = y[1:7]; T = y[7]
    rho = rho_ext + (ALPHA_DOPPLER * (T - T0) if feedback else 0.0)
    dn = (rho - BETA) / ELL * n + (LAMBDA_I * Ci).sum()
    dCi = BETA_I / ELL * n - LAMBDA_I * Ci
    dT = (max(n, 0.0) * WTH - UA * (T - TCOOL)) / MCP
    return np.concatenate(([dn], dCi, [dT]))

def cinetica_reactor(rho_pcm=100.0, feedback=True):
    rho_ext = rho_pcm * 1e-5
    y0 = np.concatenate(([1.0], BETA_I / (ELL * LAMBDA_I), [T0]))
    if rho_ext >= BETA:      t_end, regimen, col = 0.5, "PRONTO-CRÍTICO ⚠", "#C0392B"
    elif rho_ext > 0:        t_end, regimen, col = 60.0, "supercrítico retardado", "#2554E8"
    elif rho_ext == 0:       t_end, regimen, col = 60.0, "crítico (estable)", "#27AE60"
    else:                    t_end, regimen, col = 120.0, "subcrítico (decae)", "#333333"
    sol = solve_ivp(lambda t, y: _kin(t, y, rho_ext, feedback), (0, t_end), y0,
                    method="BDF", rtol=1e-7, atol=1e-9, dense_output=True)
    tt = np.linspace(0, t_end, 400); nn = sol.sol(tt)[0]
    fig, ax = plt.subplots(figsize=(9, 4.6))
    ax.plot(tt, nn, color=col, lw=2)
    ax.set_xlabel("tiempo [s]"); ax.set_ylabel("potencia normalizada  n(t)")
    ax.set_title(f"ρ = {rho_pcm:.0f} pcm  →  {regimen}"
                 f"   (β = {BETA*1e5:.0f} pcm{', con Doppler' if feedback else ''})")
    ax.grid(alpha=0.3)
    if 0 < rho_ext < BETA:
        salto = BETA / (BETA - rho_ext)
        ax.axhline(salto, ls=":", color="#C0392B")
        ax.text(t_end*0.02, salto*1.02, f"salto prompt β/(β−ρ) = {salto:.2f}",
                color="#C0392B", fontsize=9)
    plt.tight_layout(); plt.show()
    print(f"n final = {nn[-1]:.3g} · régimen: {regimen}")

interact(cinetica_reactor,
         rho_pcm=FloatSlider(100, min=-300, max=800, step=20, description="ρ [pcm]"),
         feedback=Dropdown(options=[("con Doppler", True), ("sin realimentación", False)],
                           value=True, description="realim."));


interactive(children=(FloatSlider(value=100.0, description='ρ [pcm]', max=800.0, min=-300.0, step=20.0), Dropd…

### 3.4 Fusión: barrera de Coulomb y efecto túnel (factor de Gamow)
> Los núcleos, ambos con carga +1, se repelen; solo el **túnel cuántico** permite la fusión, con probabilidad $\propto e^{-\sqrt{E_G/E}}$.


In [6]:
# WIDGET 4 — barrera de Coulomb y probabilidad de túnel (Gamow) D-T
def barrera_gamow(E_colision_keV=20.0):
    e2_4pieps = 1.44e-6      # keV·m (e²/4πε₀ en keV·metro → 1.44 MeV·fm = 1.44e-6 keV·m... usamos fm)
    # trabajamos en femtómetros: V_C(r) = 1.44 MeV·fm / r[fm]  (Z1=Z2=1)
    r = np.linspace(1, 200, 400)      # fm
    V_C = 1440.0 / r                  # keV  (1.44 MeV·fm = 1440 keV·fm)
    r_nuc = 5.0                        # radio de contacto D-T ~5 fm
    Vmax = 1440.0 / r_nuc
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
    ax1.plot(r, V_C, color="#C0392B", lw=2)
    ax1.axhline(E_colision_keV, ls="--", color="#2554E8")
    ax1.text(120, E_colision_keV*1.1, f"E colisión = {E_colision_keV:.0f} keV", color="#2554E8")
    # pozo nuclear
    ax1.plot([0, r_nuc, r_nuc], [-2000, -2000, Vmax], color="#333", lw=2)
    ax1.fill_between(r, E_colision_keV, V_C, where=(V_C > E_colision_keV), color="#C0392B", alpha=0.12)
    ax1.set_xlim(0, 200); ax1.set_ylim(-2200, Vmax*1.1)
    ax1.set_xlabel("separación r [fm]"); ax1.set_ylabel("energía potencial [keV]")
    ax1.set_title(f"Barrera de Coulomb (pico ≈ {Vmax:.0f} keV)"); ax1.grid(alpha=0.3)
    # probabilidad de túnel vs energía
    E = np.linspace(1, 200, 300)
    E_G = 1180.0    # energía de Gamow efectiva D-T [keV] (ajuste didáctico)
    P_tunel = np.exp(-np.sqrt(E_G / E))
    ax2.plot(E, P_tunel, color="#27AE60", lw=2)
    ax2.axvline(E_colision_keV, ls="--", color="#2554E8")
    ax2.scatter([E_colision_keV], [np.exp(-np.sqrt(E_G/E_colision_keV))], s=80,
                color="#2554E8", edgecolors="k", zorder=5)
    ax2.set_xlabel("energía de colisión [keV]"); ax2.set_ylabel("prob. de túnel  ∝ $e^{-\\sqrt{E_G/E}}$")
    ax2.set_title("El túnel cuántico crece con la energía"); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"A {E_colision_keV:.0f} keV la barrera (~{Vmax:.0f} keV) NO se supera por energía: "
          f"la fusión ocurre por efecto túnel (prob. relativa {np.exp(-np.sqrt(E_G/E_colision_keV)):.2e}).")

interact(barrera_gamow,
         E_colision_keV=FloatSlider(20, min=1, max=150, step=1, description="E colisión [keV]"));


interactive(children=(FloatSlider(value=20.0, description='E colisión [keV]', max=150.0, min=1.0, step=1.0), O…

### 3.5 Reactividad y potencia de fusión del plasma
> Reproduce las gráficas de $\langle\sigma v\rangle(T)$ y de potencia volumétrica $P_f=n_D\,n_T\,\langle\sigma v\rangle\,Q$.


In [7]:
# WIDGET 5 — reactividad ⟨σv⟩ y potencia de fusión vs temperatura del plasma
def plasma_fusion(n20=1.0, T_marca_keV=15.0):
    n_D = n_T = n20 * 1e20                     # partículas/m³
    T = np.linspace(1, 40, 400)
    sv = sigma_v_DT(T)
    Q_J = Q_FUS_MeV * J_per_MeV
    P_f = n_D * n_T * sv * Q_J                 # W/m³
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
    ax1.plot(T, sv, color="#C0392B", lw=2)
    ax1.axvline(T_marca_keV, ls="--", color="#555")
    ax1.set_xlabel("T [keV]"); ax1.set_ylabel(r"$\langle\sigma v\rangle$ [m³/s]")
    ax1.set_title("Reactividad Maxwelliana D-T"); ax1.grid(alpha=0.3)
    ax2.plot(T, P_f, color="#2554E8", lw=2)
    ax2.axvline(T_marca_keV, ls="--", color="#555")
    ax2.set_yscale("log"); ax2.set_xlabel("T [keV]"); ax2.set_ylabel("$P_f$ [W/m³]")
    ax2.set_title(f"Potencia de fusión (n = {n20:.1f}×10²⁰ m⁻³)")
    ax2.grid(alpha=0.3, which="both")
    sv0 = sigma_v_DT(T_marca_keV); pf0 = n_D * n_T * sv0 * Q_J
    print(f"A T = {T_marca_keV:.0f} keV:  ⟨σv⟩ = {sv0:.2e} m³/s,  P_f = {pf0:.2e} W/m³ "
          f"({pf0/1e6:.2f} MW/m³). Comparar con criterio de ignición (~10 keV).")
    plt.tight_layout(); plt.show()

interact(plasma_fusion,
         n20=FloatSlider(1.0, min=0.2, max=3.0, step=0.1, description="n [10²⁰/m³]"),
         T_marca_keV=FloatSlider(15, min=2, max=40, step=1, description="T marca [keV]"));


interactive(children=(FloatSlider(value=1.0, description='n [10²⁰/m³]', max=3.0, min=0.2), FloatSlider(value=1…

### 3.6 La comparación magna: por reacción, por nucleón y por kilogramo
> Además del desglose por reacción de la fuente, añadimos la comparación **por kilogramo** frente a la combustión química de las Lecciones 4-5.


In [8]:
# WIDGET 6 — comparación fisión vs fusión (vs combustión química)
def comparacion(base="por kilogramo"):
    fig, ax = plt.subplots(figsize=(9, 4.8))
    if base == "por reacción":
        datos = {"Fisión U-235": E_FIS_MeV, "Fusión D-T": Q_FUS_MeV}
        ax.bar(list(datos), list(datos.values()), color=["#C0392B", "#27AE60"])
        ax.set_ylabel("MeV por reacción")
        for i, v in enumerate(datos.values()):
            ax.text(i, v, f"{v:.1f} MeV", ha="center", va="bottom")
        ax.set_title("Energía por reacción: la fisión libera ~11× más POR EVENTO")
    elif base == "por nucleón":
        datos = {"Fisión U-235": E_FIS_MeV/M_U235, "Fusión D-T": Q_FUS_MeV/(M_D+M_T)}
        ax.bar(list(datos), list(datos.values()), color=["#C0392B", "#27AE60"])
        ax.set_ylabel("MeV por nucleón")
        for i, v in enumerate(datos.values()):
            ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
        ax.set_title("Energía por nucleón: la fusión gana (~3.5× la fisión)")
    else:  # por kilogramo (escala log, incluye química)
        E_fis = energia_por_kg(E_FIS_MeV, M_U235) / 1e12          # TJ/kg
        E_fus = energia_por_kg(Q_FUS_MeV, M_D + M_T) / 1e12       # TJ/kg
        E_qui = LHV_CH4_MJkg / 1e6                                 # TJ/kg
        datos = {"Combustión\nCH₄": E_qui, "Fisión\nU-235": E_fis, "Fusión\nD-T": E_fus}
        ax.bar(list(datos), list(datos.values()), color=["#7F8C8D", "#C0392B", "#27AE60"])
        ax.set_yscale("log"); ax.set_ylabel("energía específica [TJ/kg]")
        for i, v in enumerate(datos.values()):
            ax.text(i, v, f"{v:.4g} TJ/kg", ha="center", va="bottom")
        ax.set_title("Energía POR KILOGRAMO: lo nuclear es ~10⁶× la química; la fusión supera a la fisión")
    ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

interact(comparacion,
         base=Dropdown(options=["por reacción", "por nucleón", "por kilogramo"],
                       value="por kilogramo", description="comparar"));


interactive(children=(Dropdown(description='comparar', index=2, options=('por reacción', 'por nucleón', 'por k…

## 4. Resultados de la investigación original reproducidos
Reproducimos con nuestro código los valores clave que reporta el proyecto estudiantil y cuantificamos la **desviación**.


In [9]:
# Celda de verificación / reproducción de la fuente
print("=== FISIÓN U-235 ===")
E_kg_fis = energia_por_kg(E_FIS_MeV, M_U235) / 1e12
print(f"Energía específica: {E_kg_fis:.1f} TJ/kg  (fuente: ~82 TJ/kg)")
e1 = abs(E_kg_fis - 82) / 82 * 100

# consumo a 3000 MWt (fisión total, sin recorte por fracción útil, como en la fuente)
E_f = E_FIS_MeV * J_per_MeV
R_f = 3000e6 / E_f
burn_dia = R_f * (M_U235/1000/NA) * 86400
print(f"Fisiones/s @3000 MWt: {R_f:.3e}  ·  consumo U-235: {burn_dia:.2f} kg/día")
# referencia independiente de literatura: ~1.05 g de U-235 por MWd térmico
ref_lit = 1.05e-3 * 3000
e2 = abs(burn_dia - ref_lit) / ref_lit * 100
print(f"  vs literatura (~1.05 g/MWd → {ref_lit:.2f} kg/día): desviación {e2:.1f} %")

print("\n=== FUSIÓN D-T ===")
E_kg_fus = energia_por_kg(Q_FUS_MeV, M_D + M_T) / 1e12
print(f"Energía específica: {E_kg_fus:.0f} TJ/kg  ·  razón fusión/fisión = {E_kg_fus/E_kg_fis:.1f}×")
sv15 = sigma_v_DT(15.0)
print(f"Reactividad ⟨σv⟩(15 keV) = {sv15:.3e} m³/s  (fórmula de la fuente)")

print("\n=== CINÉTICA DEL REACTOR ===")
print(f"β total (6 grupos Keepin) = {BETA*1e5:.0f} pcm  (fuente: 650 pcm)")
e3 = abs(BETA*1e5 - 650) / 650 * 100
# salto prompt analítico vs régimen
rho = 0.002
print(f"Salto prompt β/(β−ρ) con ρ=200 pcm = {BETA/(BETA-rho):.3f}")

for nombre, err in [("E_kg fisión vs fuente", e1), ("consumo vs literatura", e2), ("β vs Keepin", e3)]:
    assert err < 5, f"Desviación excesiva en {nombre}: {err:.1f} %"
print(f"\n✔ Reproducción validada: desviaciones {e1:.1f} %, {e2:.1f} %, {e3:.1f} % (todas < 5 %)")


=== FISIÓN U-235 ===
Energía específica: 82.1 TJ/kg  (fuente: ~82 TJ/kg)
Fisiones/s @3000 MWt: 9.362e+19  ·  consumo U-235: 3.16 kg/día
  vs literatura (~1.05 g/MWd → 3.15 kg/día): desviación 0.2 %

=== FUSIÓN D-T ===
Energía específica: 338 TJ/kg  ·  razón fusión/fisión = 4.1×
Reactividad ⟨σv⟩(15 keV) = 6.550e-23 m³/s  (fórmula de la fuente)

=== CINÉTICA DEL REACTOR ===
β total (6 grupos Keepin) = 650 pcm  (fuente: 650 pcm)
Salto prompt β/(β−ρ) con ρ=200 pcm = 1.444

✔ Reproducción validada: desviaciones 0.1 %, 0.2 %, 0.0 % (todas < 5 %)


## 6. Créditos y cita
Este módulo adapta, con fines docentes, el proyecto final de curso de sus autores estudiantiles. Los modelos de fisión, fusión y cinética puntual, y los valores de validación, provienen de su trabajo. La cita BibTeX está en la celda siguiente.


---
### Referencias del proyecto fuente
- Duderstadt, J. J. & Hamilton, L. J. (1976). *Nuclear Reactor Analysis*. Wiley.
- Lamarsh, J. R. & Baratta, A. J. (2001). *Introduction to Nuclear Engineering*, 3ª ed. Prentice Hall.
- Wesson, J. (2011). *Tokamaks*, 4ª ed. Oxford — reactividad D-T y criterio de Lawson.
- Keepin, G. R. (1965). *Physics of Nuclear Kinetics*. Addison-Wesley — parámetros de neutrones retardados.
